> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — mvGWAS canonical xarray result

In [ ]:
from pathlib import Path
import tempfile, numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR",vigor.__version__)
cfg=vigor.simulation_scenario("complex"); cfg.update(dict(heritability=.55,compute_grms=False))
sim=vigor.simulate_genomic_dataset(n_samples=300,n_variants=800,n_traits=4,seed=42,genotype_kwargs={"n_chromosomes":4,"generations":100},trait_kwargs=cfg)
from vigor.testutils import write_plink
with tempfile.TemporaryDirectory() as td:
    prefix=str(Path(td)/"complex")
    vi=vigor.result_table(sim, "variant_info").reindex(sim.genotypes.to_pandas().columns)
    write_plink(prefix,sim.genotypes.to_pandas().to_numpy(float),list(map(str,sim.genotypes.to_pandas().index)),vi.chrom.astype(str).to_numpy(),vi.pos.to_numpy(int))
    gwas=vigor.mvGWAS(prefix,sim.phenotypes.to_pandas(),chroms=["1"],model="add-dom",verbose=False)
    print(gwas)
    print("dims",dict(gwas.sizes),"variables",list(gwas.data_vars)[:8])
    top=gwas.neglog_p.max("trait",skipna=True).sortby(gwas.neglog_p.max("trait",skipna=True),ascending=False).isel(variant=slice(0,5))
    print(top)
